In [ ]:
# Cell 1: Installaltion - to prepare collab for python code

!pip install anthropic pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 18.3 MB/s eta 0:00:00


In [ ]:
# Cell 2: Upload file, mount Google Drive, and set config

from google.colab import files, drive
import os, pandas as pd, numpy as np
import anthropic, json, re, time
import getpass

# Upload hh_subset_0.csv (or subset) when the file picker appears
# currenlty code only allows one file to be uploaded as subset 1 adn2 were not included
print("Upload your subset CSV: hh_subset_0.csv")
uploaded = files.upload()
for fname in uploaded.keys():
    print(f"  Uploaded: {fname}  ({os.path.getsize(fname):,} bytes)")

# MOUNT GOOGLE DRIVE
# Checkpoints are written straight to Drive below, not to the Colab VM's
# local disk, so a disconnect no longer costs the run its progress.
# Authorisation will be requested the first time each session runs
# click through when prompted.
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/HH_RLHF_pipeline/checkpoints"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}")
print("(rename DRIVE_DIR above if you already have a folder for this project)")

# CONFIG, only these two lines change between subsets
SUBSET_FILE = "hh_subset_0.csv"  # can swap to hh_subset_1.csv / hh_subset_2.csv if needed
SUBSET_NAME = "hh_subset_0"        # used as prefix for all output filenames


# getpass.getpass(prompt) shows the prompt text on screen and reads a
# hidden input for the return value, the key must be TYPED when
# asked, do not put inside the parentheses.
ANTHROPIC_API_KEY = getpass.getpass("Paste your Anthropic API key and press Enter: ")

CHECKPOINT_EVERY = 100   # write progress to Drive every 100 rows
N_SAMPLE         = None  # None = process the full subset
                          # set to e.g. 250 for a budget-constrained sample run first
RANDOM_SEED      = 42    # used only if N_SAMPLE is not None

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

Upload your subset CSV: hh_subset_0.csv


Saving hh_subset_0.csv to hh_subset_0.csv
  Uploaded: hh_subset_0.csv  (48,434,005 bytes)
Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/HH_RLHF_pipeline/checkpoints
(rename DRIVE_DIR above if you already have a folder for this project)
Paste your Anthropic API key and press Enter: ··········


In [ ]:
# Cell 3: Load and inspect the subset

df = pd.read_csv(SUBSET_FILE)

# HH-RLHF rows have no natural unique ID (CMV has comment_id), create one
# from row position so the checkpoint/resume logic in Cell 6 has something
# stable to track. Must happen before any sampling/filtering.
df = df.reset_index(drop=True)
df["row_id"] = df.index

# Quick sanity check before committing to a full API run
# Figures below are what hh_subset_0.csv actually contains (checked directly):
# rows: 28,683 | topics: 48 | source_subset: helpful_rej 13,225 / helpful_base
# 9,062 / helpful_online 6,396 | multi-turn ~100%
# If printed numbers look very different, double check the upload.
print(f"Subset file     : {SUBSET_FILE}")
print(f"Rows            : {len(df):,}")
print(f"Topics          : {df['topic'].nunique()}")
print(f"Source subsets  :")
print(df['source_subset'].value_counts().to_string())
print(f"Multi-turn share: {(~df['is_single_turn']).mean():.1%} (expected ~100%)")
df.head(3)

Subset file     : hh_subset_0.csv
Rows            : 28,683
Topics          : 48
Source subsets  :
source_subset
helpful_rej       13225
helpful_base       9062
helpful_online     6396
Multi-turn share: 100.0% (expected ~100%)


,source_subset,human_context,first_human_turn,chosen_response,rejected_response,n_human_turns,is_single_turn,earned_delta,hh_subset,topic,subset,row_id
0,helpful_rej,Human: I'm moving to Utah next month and I'm c...,I'm moving to Utah next month and I'm curious ...,There's actually a lot of great climbing in th...,"St. George is a great city for rock climbing, ...",4,False,1,subset_B,8,0,0
1,helpful_rej,Human: What is an easy to make cake frosting?\...,What is an easy to make cake frosting?,"You don’t need to cook it, but you do need to ...","No, it won’t be cooked, and you can eat it rig...",3,False,1,subset_B,0,0,1
2,helpful_rej,Human: What age do I start talking to kids abo...,What age do I start talking to kids about pube...,One simple starting place is to just go throug...,"Just to get started, you can explain to your k...",4,False,1,subset_B,21,0,2


In [ ]:
# Cell 4: Define extraction prompt and functions

import anthropic as anthropic_module  # needed to catch API-specific exceptions

# Adapted from CMV EXTRACTION_PROMPT, already sanity-checked on 5 rows
# from different topics, do not modify without re-testing:
#   1. Framing changed from Reddit debate comment to AI preference pair
#   2. Input slots changed: human_context + chosen_response + rejected_response
#      (there is no "OP post" in HH-RLHF)
#   3. Extraction question reframed around the preference contrast -- the rule
#      must explain WHY the chosen response was preferred over the rejected one
#   4. Explicit DO NOT extract items added for response formatting, length, and
#      communication style, to avoid the known RLHF bias where a model picks
#      up "give complete/well-formatted answers" instead of the moral/epistemic norm
#      we are trying to uncover. 
#      (NOTE: the debug run showed this instruction does not fully catch every
#      rule)
#      (NOTE: Further testing showed that instruction 4 had no significant effect on
#      structural rules being extracted)

EXTRACTION_PROMPT = """\
You are analysing a conversation between a human and an AI assistant. \
A human evaluator was shown two possible AI responses to this conversation \
and chose one as better than the other.

Conversation context:

{human_context}

Preferred response (chosen by human evaluator):

{chosen_response}

Non-preferred response (rejected by human evaluator):

{rejected_response}

Your task: extract every rule, norm, principle, or heuristic that is stated or \
clearly implied by this preference — generalizable statements about HOW people \
expect others to reason, behave, or make decisions in situations like this one.

Focus on what the preference REVEALS about human expectations, not on describing \
the quality of the responses. Ask: what does it mean that a human preferred one \
response over the other? What rule were they applying?

Here are some examples of rules:

1. Domain-specific rules of thumb — practical guidelines for a type of situation \
(e.g. "when giving medical advice, always recommend consulting a professional")

2. Pragmatic interaction norms — how one should behave toward others \
(e.g. "acknowledge the emotional dimension of a request before giving practical advice")

3. Moral heuristics — ethical principles invoked to justify or critique a choice \
(e.g. "you should not encourage someone to take actions that could harm others")

4. Reasoning patterns — general decision strategies or appeals to fairness, \
precedent, or consequences (e.g. "when uncertain, acknowledge the limits of your knowledge")

Rules to extract:

- Generalizable principles, not facts specific only to this conversation

- Implied by the preference contrast — what does choosing A over B reveal?

- Each rule must be understandable to an intelligent adult who has not read \
the source conversation

Do NOT extract:

- Rules about response formatting, length, or completeness

- Rules about communication style or tone alone

- Pure factual statements about the specific topic

- Rules that only make sense within this specific conversation

Return ONLY a raw JSON array of strings with no markdown, no code fences, no \
explanation. Each string should be one concise rule in general form. Example:

["When giving advice on sensitive topics, prioritise safety over directness.", \
"Acknowledge uncertainty rather than stating guesses as facts."]

If no rules can be extracted, return an empty list."""


def parse_json_response(text: str) -> list:
    """
    Clean and parse the model's raw text output into a Python list.
    The model sometimes wraps its response in markdown code fences (```json ... ```)
    even when instructed not to -- the regex strips those before parsing.
    """
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)  # remove opening fence
    text = re.sub(r"\s*```$", "", text)            # remove closing fence
    return json.loads(text.strip())


def extract_rules(human_context: str, chosen_response: str,
                  rejected_response: str, debug: bool = False) -> list:
    """
    Send one HH-RLHF row (context + chosen + rejected) to Claude Haiku and
    return a list of extracted rule strings. Returns an empty list if the
    model finds nothing extractable or if parsing fails.

    Retry logic handles transient server errors (500) which can occur on long runs.
    Uses exponential backoff: waits 5s, then 10s, then 20s before giving up.
    If all retries fail, returns an empty list so the run continues rather than crashes.
    """
    prompt = EXTRACTION_PROMPT.format(
        human_context     = str(human_context)[:1500],
        chosen_response   = str(chosen_response)[:800],
        rejected_response = str(rejected_response)[:800],
    )

    MAX_RETRIES = 3
    BACKOFF     = [5, 10, 20]  # seconds to wait before each retry attempt

    for attempt in range(MAX_RETRIES + 1):
        try:
            msg = client.messages.create(
                model      = "claude-haiku-4-5-20251001",
                max_tokens = 1024,
                messages   = [{"role": "user", "content": prompt}],
            )

            raw = msg.content[0].text

            if debug:
                print("RAW RESPONSE:", repr(raw))

            try:
                return parse_json_response(raw)
            except Exception as e:
                # JSON parse failure -- return empty, don't crash the run
                if debug:
                    print(f"Parse error: {e}")
                return []

        except anthropic_module.InternalServerError as e:
            # Transient 500 error from Anthropic's servers -- wait and retry
            if attempt < MAX_RETRIES:
                wait = BACKOFF[attempt]
                print(f"  [500 error] Retrying in {wait}s... (attempt {attempt + 1}/{MAX_RETRIES})")
                time.sleep(wait)
            else:
                print(f"  [500 error] All retries failed for row -- skipping.")
                return []

        except anthropic_module.RateLimitError:
            # Rate limit hit -- wait longer than usual and retry
            if attempt < MAX_RETRIES:
                wait = BACKOFF[attempt] * 2
                print(f"  [Rate limit] Waiting {wait}s before retry...")
                time.sleep(wait)
            else:
                print(f"  [Rate limit] All retries failed -- skipping.")
                return []

        except anthropic_module.APIConnectionError:
            # Network-level error -- wait and retry
            if attempt < MAX_RETRIES:
                wait = BACKOFF[attempt]
                print(f"  [Connection error] Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  [Connection error] All retries failed -- skipping.")
                return []

print("Prompt and functions loaded.")

Prompt and functions loaded.


In [ ]:
# Cell 5: Debug check on first row

# Always run this before the full extraction
# Read the output carefully, check rules look genuinely generalisable
# and not topic-specific, formatting-related, or too opaque without context
#
# RECOMMENDATION: run this on more than one row (loop over df.iloc[0:8] from
# different topics) before Cell 6

sample = df.iloc[0]

print("=== CONTEXT (first 300 chars) ===")
print(str(sample["human_context"])[:300])
print()
print("=== CHOSEN RESPONSE (first 200 chars) ===")
print(str(sample["chosen_response"])[:200])
print()
print("=== REJECTED RESPONSE (first 200 chars) ===")
print(str(sample["rejected_response"])[:200])
print()
print("=== EXTRACTED RULES ===")
test_rules = extract_rules(
    sample["human_context"],
    sample["chosen_response"],
    sample["rejected_response"],
    debug=True   # prints raw model output for JSON-format verification
)
for r in test_rules:
    print(f"  * {r}")
print(f"\nTotal rules extracted: {len(test_rules)}")

=== CONTEXT (first 300 chars) ===
Human: I'm moving to Utah next month and I'm curious about the best cities in Utah for rock climbing.

Assistant: I think there are a lot of fantastic climbing areas to explore, especially in Zion National Park. But I also know a few areas nearby that are really great to climb if you have some extra

=== CHOSEN RESPONSE (first 200 chars) ===
There's actually a lot of great climbing in the area around St. George. It's possible to climb near Bryce Canyon and Zion National Park, and Coral Pink Sand Dunes State Park and Maze District are grea

=== REJECTED RESPONSE (first 200 chars) ===
St. George is a great city for rock climbing, and it’s close to many amazing rock formations like the sandstone fins in Cedar Mesa, the red rocks around Goblin Valley State Park, and the red rocks in 

=== EXTRACTED RULES ===
RAW RESPONSE: '```json\n[\n  "When listing geographic locations or attractions, prioritize accuracy and verify that they are actually associated with 

In [ ]:
# Cell 6: Full extraction run with checkpoint/resume (saved to Drive)

output_file     = os.path.join(DRIVE_DIR, f"{SUBSET_NAME}_rules.csv")             # final output
checkpoint_file = os.path.join(DRIVE_DIR, f"{SUBSET_NAME}_rules_checkpoint.csv")  # rolling save, on Drive

# RESUME LOGIC
# If a checkpoint exists on Drive from a previous run (even a
# previous session, not just this runtime), load it and pick up from
# where it left off. Tracks progress by row_id (created in Cell 3).
if os.path.exists(checkpoint_file):
    existing      = pd.read_csv(checkpoint_file)
    processed_ids = set(existing["row_id"].unique())
    all_rules     = existing.to_dict("records")
    print(f"Resuming from Drive checkpoint: {len(processed_ids):,} rows already done")
else:
    processed_ids = set()
    all_rules     = []
    print("Starting fresh")

# Filter out already-processed rows before iterating
to_process = df[~df["row_id"].isin(processed_ids)].reset_index(drop=True)
print(f"Rows remaining: {len(to_process):,}")
print()

for i, row in to_process.iterrows():

    # Sends this row to the API and get back a list of rule strings
    rules = extract_rules(
        row["human_context"],
        row["chosen_response"],
        row["rejected_response"],
    )

    # Each rule becomes its own row in the output, inheriting source metadata
    for r in rules:
        all_rules.append({
            "row_id"           : row["row_id"],
            "topic"            : row["topic"],
            "source_subset"    : row["source_subset"],
            "hh_subset"        : row["hh_subset"],
            "human_context"    : row["human_context"],
            "chosen_response"  : row["chosen_response"],
            "rejected_response": row["rejected_response"],
            "rule"             : r,
        })

    # Save checkpoint to Drive every CHECKPOINT_EVERY rows
     # If Colab disconnects or the runtime gets recycled, at most
    # CHECKPOINT_EVERY calls worth of work is lost, and it's
    # recoverable even in a brand new session since it lives on
    # Drive, not the VM disk.
    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_rules).to_csv(checkpoint_file, index=False)
        print(f"  {i+1}/{len(to_process)} rows | {len(all_rules):,} rules so far | saved to Drive")

    # Rate limiting,  pause between API calls to avoid hitting request limits
    time.sleep(0.3)

# FINAL SAVE
rules_df.to_csv(output_file, index=False)

print()
print("Done.")
print(f"Rows processed     : {rules_df['row_id'].nunique():,}")
print(f"Rules extracted    : {len(rules_df):,}")
print(f"Avg rules/row      : {len(rules_df)/rules_df['row_id'].nunique():.2f}")
print(f"Empty rules        : {(rules_df['rule'].str.strip() == '').sum()}")
print(f"Null rules         : {rules_df['rule'].isna().sum()}")
print(f"Saved to Drive at  : {output_file}")

Starting fresh
Rows remaining: 28,683



In [ ]:
# # Cell 7: Download output and spot check quality
from google.colab import files

# Trigger browser download of the final output file (backup copy,
# the primary copy already lives on Drive at output_file from Cell 6)
files.download(output_file)

# Read 10 random rules as a sanity check on output quality.
# If rules look too topic-specific, formatting-related, or opaque,
# revisit the prompt.
print("=== SAMPLE RULES (random 10) ===")
for r in rules_df["rule"].sample(min(10, len(rules_df)), random_state=42):
    print(f"  * {r}")

print()

# hh_subset_0.csv only has numeric "topic" IDs, not the "topic_label"
# column CMV's rules_df has, grouping by ID here instead of
# assuming a label exists.
print("=== RULES PER TOPIC ID (top 10 by count) ===")
print(
    rules_df.groupby("topic")["rule"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)